# Gold-layer customer profile

This notebook creates a Gold-layer table 
"banking.gold.customer_profile".

The goal is to build a complete customer profile
by combining:
- customer data
- account data
- transaction data
- branch information
- credit bureau information

The final table is used for:
- analytics
- dashboards
- customer segmentation
- risk analysis

In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.customer_profile AS

--Account Aggregation
--Metrics:
    --total_accounts:
    --total_balance:
--Source: banking.silver.accounts

WITH account_agg AS (
SELECT
    customer_id,
    COUNT(account_id) AS total_accounts,
    SUM(balance) AS total_balance
FROM banking.silver.accounts
GROUP BY customer_id
),

--Transaction Aggregation
--Metrics:
    --total_transactions
    --total_transaction_amount

txn_agg AS (
SELECT
    a.customer_id,
    COUNT(t.txn_id) AS total_transactions,
    SUM(t.amount) AS total_transaction_amount
FROM banking.silver.transactions t
JOIN banking.silver.accounts a
    ON t.account_id = a.account_id
GROUP BY a.customer_id
),

--Get Latest Credit Report
credit_latest AS (
SELECT *
FROM (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY bureau_pull_date DESC
        ) AS rn
    FROM banking.silver.credit_bureau_reports
    )
WHERE rn = 1
)

--Build Customer Dataset 
--Included data:
    -- customer information
    -- branch information
    -- account metrics
    -- transaction metrics
    -- credit bureau metrics
SELECT
    c.customer_id,
    CONCAT(c.first_name,' ',c.last_name) AS customer_name,
    b.branch_name,
    COALESCE(a.total_accounts,0) AS total_accounts,
    COALESCE(a.total_balance,0) AS total_balance,
    COALESCE(t.total_transactions,0) AS total_transactions,
    COALESCE(t.total_transaction_amount,0) AS total_transaction_amount,
    cr.credit_score,
    cr.risk_grade,
    cr.external_active_loans,
    cr.external_overdue_amount,
    CASE
        WHEN a.total_balance >= 500000 THEN 'HIGH_VALUE'
        WHEN a.total_balance >= 100000 THEN 'MEDIUM_VALUE'
        ELSE 'LOW_VALUE'
    END AS customer_segment
FROM banking.silver.customers c
--Join Account Aggregation
LEFT JOIN account_agg a
    ON c.customer_id = a.customer_id
--Join Transaction Aggregation
LEFT JOIN txn_agg t
    ON c.customer_id = t.customer_id
--Join Branch Table
LEFT JOIN banking.silver.branches b
    ON c.branch_code = b.branch_code
--Join Latest Credit Report
LEFT JOIN credit_latest cr
    ON c.customer_id = cr.customer_id



## Validate Table Creation
Counts the number of rows created in the Gold table.
Used for:
- validation
- workflow checks

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.customer_profile
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))